# California Yoga Studio Reviews
## Notebook 6: Classifier - INCOMPLETE
*2026-05*

This notebook documents the design of a logistic regression classifier
intended to predict high (4-5★) vs low (1-3★) ratings from NLP-derived
features — VADER sentiment scores, review text length, dominant LDA topic,
and studio-level metadata.

The classifier was not trained to completion due to two fundamental dataset
constraints identified during feature matrix construction.

**Input:** `yoga_reviews_topics.pkl`, `yoga_studios_meta.pkl`

### Why the classifier was not built and future plans

The classifier was not built because of severe class imbalance and limited negative review signal. The analysis will later be extended with a broader dataset from more USA states with strong yoga leaning to address these constraints.

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, roc_auc_score, roc_curve)
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

PROJECT_DIR = '/content/gdrive/MyDrive/YogaStudioReviews'

reviews = pd.read_pickle(os.path.join(PROJECT_DIR, 'yoga_reviews_topics.pkl'))
yoga_meta = pd.read_pickle(os.path.join(PROJECT_DIR, 'yoga_studios_meta.pkl'))

print(f'Loaded {len(reviews):,} reviews')
print(f'Columns: {reviews.columns.tolist()}')

Mounted at /content/gdrive
Loaded 10,547 reviews
Columns: ['user_id', 'name', 'time', 'rating', 'text', 'resp', 'gmap_id', 'date', 'text_length', 'pic_count', 'vader_pos', 'vader_neg', 'vader_neu', 'vader_compound', 'sentiment', 'dominant_topic_all']


In [ ]:
reviews['has_photo'] = reviews['pic_count'] > 0

In [ ]:
# Binary target — high (4-5 star) vs low (1-3 star)
# We use 1-3 as "low" rather than just 1-2 because:
# - 3-star reviews express genuine mixed/negative sentiment
# - gives better class balance than 1-2 only

reviews['high_rating'] = (reviews['rating'] >= 4).astype(int)

print('=== Target variable distribution ===')
counts = reviews['high_rating'].value_counts()
print(f'High rating (4-5★): {counts[1]:,} ({counts[1]/len(reviews)*100:.1f}%)')
print(f'Low rating (1-3★):  {counts[0]:,} ({counts[0]/len(reviews)*100:.1f}%)')
print(f'\nClass imbalance ratio: {counts[1]/counts[0]:.1f}:1')

=== Target variable distribution ===
High rating (4-5★): 10,115 (95.9%)
Low rating (1-3★):  432 (4.1%)

Class imbalance ratio: 23.4:1


In [ ]:
# Merge studio-level features into reviews
reviews_enriched = reviews.merge(
    yoga_meta[['gmap_id', 'precise_avg', 'mean_compound',
               'open_weekend', 'status']],
    on='gmap_id', how='left'
)

# Build feature matrix
features = pd.DataFrame({
    # VADER sentiment features
    'vader_compound':  reviews_enriched['vader_compound'],
    'vader_pos':       reviews_enriched['vader_pos'],
    'vader_neg':       reviews_enriched['vader_neg'],
    # Text features
    'text_length':     reviews_enriched['text_length'],
    'has_photo':       reviews_enriched['has_photo'].astype(int),
    # LDA topic features — one-hot encode dominant topic
    'dominant_topic':  reviews_enriched['dominant_topic_all'],
    # Studio-level features
    'studio_avg_rating':    reviews_enriched['precise_avg'],
    'studio_mean_compound': reviews_enriched['mean_compound'],
    'open_weekend':         reviews_enriched['open_weekend'].astype(int),
    'studio_closed':        (reviews_enriched['status'] == 'Closed').astype(int)
})

# One-hot encode dominant topic
topic_dummies = pd.get_dummies(
    features['dominant_topic'],
    prefix='topic',
    drop_first=True
)
features = pd.concat([features.drop(columns='dominant_topic'), topic_dummies], axis=1)

# Drop rows with missing values
features['target'] = reviews['high_rating']
features = features.dropna()

X = features.drop(columns='target')
y = features['target']

print(f'Feature matrix shape: {X.shape}')
print(f'Features: {X.columns.tolist()}')
print(f'Target distribution: {y.value_counts().to_dict()}')

Feature matrix shape: (8162, 16)
Features: ['vader_compound', 'vader_pos', 'vader_neg', 'text_length', 'has_photo', 'studio_avg_rating', 'studio_mean_compound', 'open_weekend', 'studio_closed', 'topic_1', 'topic_2', 'topic_3', 'topic_4', 'topic_5', 'topic_6', 'topic_7']
Target distribution: {1.0: 7883, 0.0: 279}
